# 12 — Is there a monocyte → macrophage → SPP1/TREM2-TAM trajectory, and what changes along it?

Every notebook up to here has treated the SPP1-TAM as a **state you are in or out of** — a
signature quartile, Alice's `cell_type_lvl4` call, an `M2_like_Mac` label. None of them can ask
the question this one asks:

> **Is there a continuous ordering from monocyte to SPP1/TREM2 TAM, where does it branch, and
> what turns on where along it?**

The substrate is `annotation_v4.h5ad`, which carries the whole ladder in one annotation:

| `cell_type_lvl2` | n | `cell_type_lvl3` states inside it |
|---|---|---|
| `Classical_Mono` | 25,955 | `Phagocytic_Mono` (12,339), `IFN_stimulated_Mono` (13,616) |
| `Nonclassical_Mono` | 4,570 | `Patrolling_Mono` (4,570) |
| `Intermediate_Mac` | 9,221 | `Mmp12_high_Mac` (9,221) |
| `M1_like_Mac` | 15,716 | `IFN_stimulated_Mac` (15,716) |
| `M2_like_Mac` | 23,538 | `Apoe_high_Mac` (11,097), `Arg1_high_Mac` (6,335), `Fn1_high_Mac` (6,106) |

**79,000 myeloid cells.** `cDC`/`pDC` are excluded — dendritic cells are not monocyte-derived in
the sense this trajectory means, and leaving them in buys a branch that is a lineage artifact.
Neutrophils, lymphoid and tumour are excluded for the same reason.

> ### What pseudotime is, and is not, here
> These are **endpoint tumours, dissociated once**. There is no time axis in the data, and there
> are **no spliced/unspliced counts anywhere in this project** — no `.loom`, no velocyto output,
> so RNA velocity is not available and is not coming. What scFates fits is a principal tree
> through a similarity manifold; "pseudotime" is arc length along that tree.
>
> It is a **transcriptional ordering**, not a measured differentiation time. Its *direction* is
> set entirely by where the root is put, and the root here is a **marker decision taken in §4,
> before the tree is fitted** — never a result, and never chosen to make the answer come out.

### The four ways this would fool us

Each gets a control in §6, because each produces a *positive-looking* trajectory.

1. **Library complexity.** This is the artifact that already burned this project once — `09` §5
   showed a per-sample gene-detection offset was the largest thing in the naive between-arm
   contrast, with a control pathway as the biggest mover. A trajectory is *more* vulnerable than
   a DE test, not less: "number of genes detected" is a smooth manifold axis, and monocytes and
   macrophages genuinely differ in depth, so a depth gradient and the real biology point the same
   way. → §6 control 1 measures pseudotime-vs-depth **within sample**, and re-checks the state
   ordering **inside depth quartiles**.
   > This is also why **CytoTRACE cannot be the independent root check** — CytoTRACE's
   > differentiation potential *is* a gene-count statistic, so it and the artifact are the same
   > signal. It appears in `13` as a kernel, with that caveat attached, never as the root.
2. **Batch.** There is exactly **one library per sample** (`Library` ↔ `file_name` is 1:1 over all
   29), so batch and animal are perfectly confounded. Harmony on `sample` is not optional here.
   → control 2 decomposes pseudotime variance into sample vs cell state, and reproduces the
   ordering on the object's own inherited `X_pca_harmony` with an independent method (DPT).
3. **Circularity.** The 50-gene `SPP1_TAM_sig` from `01c` is what *names* the terminal tip. Asking
   which genes rise toward that tip and answering "the signature genes" proves nothing. →
   signature genes are **excluded from every driver and module claim**, and the tip is
   re-identified with Alice's independent `cell_type_lvl4 == Spp1_pos_mac` call, which was made
   without this signature.
4. **One fit is not evidence.** The project rule everywhere else (`08b`, `09`, `10`,
   `rtAdvantage/01`) is **reproducibility across the two CAR-T backgrounds**, not a p-value. →
   control 4 refits the entire manifold and tree separately in the PSCA arms and in the TAG72
   arms and asks whether the same ordering comes back, with a label permutation for the null.

### What this notebook does *not* claim
It does not claim monocytes in this tumour become SPP1-TAMs *in vivo* — a cross-sectional
snapshot cannot show that. It claims, or fails to claim, that the transcriptomes lie on a
**reproducible one-dimensional ordering** whose two ends are monocyte and SPP1/TREM2 TAM, and it
reports what changes along it. Condition effects on that ordering are `13`'s job.

## 1. Config

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, scanpy as sc, anndata as ad, h5py
import scipy.sparse as sp
from scipy import stats
from pathlib import Path
import matplotlib as mpl, matplotlib.pyplot as plt

# ---- paths: HPC kernel or a local copy, whichever resolves first -------------------------
# This notebook runs LOCALLY — everything it needs is on the Mac (annotation_v4.h5ad, the
# pathway GMTs, the signature, the lvl4 labels). The /coh_labs entries are kept only so the same
# file also runs unchanged if it is ever moved to the cluster.
CANDIDATES = [Path("/Users/janzules/Roselab/Spatial/CAR_T"),
              Path("/coh_labs/yunroseli/Jona/CAR-T"),
              Path("/coh_labs/yunroseli/Jona/spatial/CAR-T")]
PROJ = next(p for p in CANDIDATES if p.exists())
_HPC = PROJ.as_posix().startswith("/coh_labs")
REF_DIR = (PROJ/"data/zarr/fullDataset/annotating_references") if _HPC else (PROJ/"data/sc-reference")
SC_REF  = REF_DIR/"annotation_v4.h5ad"        # 93,548 x 22,949, .X log1p, layers['counts'] raw
MAC_REF = REF_DIR/"mac_annotation_v5.h5ad"    # 48,475 macrophages; LOCAL ONLY, see LAB_PARQUET
LAB_PARQUET = PROJ/"results/spp1_analysis/mac_lvl3_lvl4_lvl5_labels.parquet"
GMT_DIR = PROJ/"data/references/pathways"
SIG_CSV = PROJ/"results/spp1_analysis/sc_spp1_tam_signature.csv"
OUT = PROJ/"results/spp1_analysis/sc_myeloid_trajectory"
FIG = OUT/"figures"; FIG.mkdir(parents=True, exist_ok=True)

# ---- the ladder ---------------------------------------------------------------------------
LADDER = ["Classical_Mono", "Nonclassical_Mono", "Intermediate_Mac", "M1_like_Mac", "M2_like_Mac"]
LVL3_ORDER = ["Phagocytic_Mono", "IFN_stimulated_Mono", "Patrolling_Mono", "Mmp12_high_Mac",
              "IFN_stimulated_Mac", "Fn1_high_Mac", "Arg1_high_Mac", "Apoe_high_Mac"]
CART = {"NoTx": "none", "CyPSCA": "PSCA", "RTCyPSCA": "PSCA",
        "CyTAG72": "TAG72", "RTCyTAG72": "TAG72"}
MONO_CLASSICAL = {"Phagocytic_Mono", "IFN_stimulated_Mono"}   # the only legitimate root states
MIN_CELLS_PER_NODE = 20       # a node with fewer cells than this cannot be the root

# ---- knobs --------------------------------------------------------------------------------
SEED, N_PCS, N_NEIGH = 0, 50, 30
import os
N_JOBS      = max(1, (os.cpu_count() or 4) // 2)   # physical/performance cores, not logical:
                            # an M3 Pro reports 12 but has 6 performance cores, and handing GAM
                            # fits to the efficiency cores makes the whole step slower, not faster
N_NODES     = 500           # principal-tree nodes (~1 per 160 cells at full size)
CYCLE_FLAG  = 0.25          # prog_cycle above this marks a milestone as a proliferation branch
N_MAP       = 1             # >1 = probabilistic pseudotime over n_map mappings (slow; 1 is fine)
PPT_SIGMA   = None          # set by the §5 sweep; None -> the sweep picks it
PPT_LAMBDA  = 100           # soft-assignment regularisation; higher = smoother tree
CELL_CAP    = None          # int -> cap cells per library (smoke test). None = all 79k
DETECT_FLOOR = 0.05         # a gene must be detected in >5% of cells to be tested along the tree
GAM_CELL_CAP = 15000        # cells used for the §7 GAM fits (None = all). See the §7 note.
N_PERM      = 2000          # label permutations for the cross-background null

# ---- palette (identical slots to notebook 11) ---------------------------------------------
S1, S2, S3 = "#2a78d6", "#eb6834", "#1baf7a"            # categorical slots 1-3 (all-pairs safe)
INK, MUTED, GRID, SURF, FADE = "#0b0b0b", "#52514e", "#e6e5e2", "#fcfcfb", "#dcdbd7"
SEQ = mpl.colors.LinearSegmentedColormap.from_list(
    "seq_blue", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])   # magnitude: one hue
DIV = mpl.colors.LinearSegmentedColormap.from_list(
    "div_br", ["#1c5cab", "#3987e5", "#f0efec", "#e34948", "#b02f2e"])     # polarity: 2 hues + gray
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "path",
                     "figure.facecolor": SURF, "axes.facecolor": SURF,
                     "axes.edgecolor": GRID, "axes.labelcolor": MUTED,
                     "xtick.color": MUTED, "ytick.color": MUTED, "font.size": 9})
pd.set_option("display.width", 220, "display.max_columns", 60)
rng = np.random.default_rng(SEED)

print(f"project   : {PROJ}")
print(f"reference : {SC_REF}")
print(f"outputs   : {OUT}")
for m in ("scanpy", "anndata"):
    import importlib.metadata as _md; print(f"{m:10s}: {_md.version(m)}")
for m in ("scFates", "palantir", "cellrank", "harmonypy"):
    import importlib.metadata as _md
    try: print(f"{m:10s}: {_md.version(m)}")
    except Exception: print(f"{m:10s}: MISSING  <- wrong kernel; use 'traj_r_py311'")

# scFates fits its GAMs through R's mgcv over rpy2 (§7-§8). Checked here rather than 40 minutes in.
try:
    from rpy2.robjects.packages import importr; importr("mgcv")
    import rpy2.robjects as _ro; print(f"rpy2+mgcv : OK ({_ro.r('R.version.string')[0]})")
except Exception as e:
    print(f"rpy2+mgcv : NOT AVAILABLE — {type(e).__name__}: {str(e)[:90]}")
    print("            §1-§6 (tree, root, pseudotime, all four controls) still run.")
    print("            §7-§8 (gene association, modules, fork) need the traj_r_py311 kernel,")
    print("            whose kernel.json sets R_HOME. If you launched it another way, set")
    print("            R_HOME=~/mamba_envs/traj_r_py311/lib/R before starting Jupyter.")

## 2. Build the myeloid object

Three things are attached to the cells here, and it matters that they come from three
**independent** places — that is what makes the terminal-state call in §5 checkable rather than
circular:

- `ct3` — Alice's `cell_type_lvl3`, the annotation. Not derived from anything in this notebook.
- `spp1_call` — Alice's `cell_type_lvl4` (`Spp1_pos_mac` / `Spp1_neg_mac`) from
  `mac_annotation_v5.h5ad`. Defined **only on the 48,475 macrophages**, so monocytes come back
  `not_in_mac_object` — which is correct, not a join failure, and is asserted below.
- `SPP1_TAM_sig` — the project's own 50-gene graded score from `01c`. `Spp1` itself is not in it.

In [ ]:
with h5py.File(SC_REF, "r") as f:
    obs = ad.io.read_elem(f["obs"]); var = ad.io.read_elem(f["var"]); X = ad.io.read_elem(f["X"])
full = ad.AnnData(X=X, obs=obs, var=var); del X

mx = float(full.X[:2000].max())
assert mx < 50, f".X max = {mx:.1f} — these are raw counts, not log1p. ABORT."
print(f"loaded {full.n_obs:,} x {full.n_vars:,} | .X max {mx:.2f} -> log1p, used as-is")

keep = full.obs["cell_type_lvl2"].astype(str).isin(LADDER).values
adata = full[keep].copy(); del full
print(f"myeloid ladder: {adata.n_obs:,} cells ({100*keep.mean():.1f}% of the object)")

o = adata.obs
o["ct2"]    = o["cell_type_lvl2"].astype(str)
o["ct3"]    = o["cell_type_lvl3"].astype(str)
o["cond"]   = o["condition"].astype(str)
o["sample"] = o["file_name"].astype(str)
o["tu"]     = o["sample"].str.extract(r"_(Tu\d)_")[0].values      # Tu1 irradiated, Tu2 abscopal
o["mouse"]  = o["sample"].str.replace(r"_Tu\d_", "_", regex=True)
o["cart"]   = o["cond"].map(CART)
o["RT"]     = o["cond"].str.startswith("RT")

# --- Alice's independent Spp1 call, macrophages only -----------------------------------------
# mac_annotation_v5.h5ad is 4.4 GB and exists only on the Mac, so the three label columns were
# extracted once to a 0.5 MB parquet that lives with the results. The h5ad is used if present.
if LAB_PARQUET.exists():
    mobs = pd.read_parquet(LAB_PARQUET); src = LAB_PARQUET.name
elif MAC_REF.exists():
    with h5py.File(MAC_REF, "r") as f:
        mobs = ad.io.read_elem(f["obs"])[["cell_type_lvl3", "cell_type_lvl4", "cell_type_lvl5"]]
    src = MAC_REF.name
else:
    raise FileNotFoundError(
        f"neither {LAB_PARQUET} nor {MAC_REF} is reachable — the independent lvl4 SPP1 call is "
        f"what keeps the terminal-state call in §5 non-circular, so this is not optional.")
o["spp1_call"]  = mobs["cell_type_lvl4"].astype(str).reindex(o.index).fillna("not_in_mac_object").values
o["isg15_call"] = mobs["cell_type_lvl5"].astype(str).reindex(o.index).fillna("not_in_mac_object").values
print(f"lvl4/lvl5 calls read from {src}")

joined_mono = (o.loc[o.ct2.str.contains("Mono"), "spp1_call"] != "not_in_mac_object").sum()
joined_mac  = (o.loc[~o.ct2.str.contains("Mono"), "spp1_call"] != "not_in_mac_object").sum()
n_mac = int((~o.ct2.str.contains("Mono")).sum())
assert joined_mono == 0, f"{joined_mono} monocytes carry an lvl4 call — the join is wrong"
assert joined_mac == n_mac, f"only {joined_mac:,}/{n_mac:,} macrophages joined — index mismatch"
print(f"lvl4 join: {joined_mac:,}/{n_mac:,} macrophages, 0 monocytes  (as designed)")

# --- the project's graded signature (01c); Spp1 is deliberately not one of its genes ----------
SIG = pd.read_csv(SIG_CSV)["gene"].tolist()
SIG_PRESENT = [g for g in SIG if g in adata.var_names]
assert "Spp1" not in SIG_PRESENT
sc.tl.score_genes(adata, SIG_PRESENT, score_name="SPP1_TAM_sig", random_state=SEED)
print(f"SPP1_TAM_sig scored from {len(SIG_PRESENT)}/{len(SIG)} signature genes")

if CELL_CAP:
    idx = (o.groupby("sample", observed=True)
             .apply(lambda d: d.sample(min(len(d), CELL_CAP), random_state=SEED).index)
             .explode().to_numpy())
    adata = adata[idx].copy(); o = adata.obs
    print(f"\n*** SMOKE TEST: capped at {CELL_CAP}/library -> {adata.n_obs:,} cells ***")

print("\n=== composition ===")
tab = (pd.crosstab(o.ct3, o.ct2).reindex(LVL3_ORDER)
         .loc[:, [c for c in LADDER if c in set(o.ct2)]])
print(tab.replace(0, "").to_string())
print(f"\nsamples: {o['sample'].nunique()} | mice: {o['mouse'].nunique()} | "
      f"conditions: {sorted(o.cond.unique())}")
print(f"\ncells per condition x site:\n{pd.crosstab(o.cond, o.tu).to_string()}")

## 3. Re-embed the myeloid compartment — and: *does clustering show the M1/M2 split?*

The object ships an `X_pca_harmony` computed over **all 93,548 cells**, where the leading
components are spent separating tumour from lymphoid from myeloid. Within-myeloid structure is
whatever is left over. So the manifold is rebuilt on the 79k myeloid cells alone. Control 2 in §6
then re-derives the ordering on the *inherited* embedding, so this choice is checkable rather
than assumed.

**Genes blocked from the HVG set, and why each one would be a real problem:**
- `PSCA` / `TAG72` — the spiked-in CAR-transgene features. They read **only** in their own arms
  and **zero in NoTx** (`project-modality-split-and-car-transgene`), so leaving them in hands the
  manifold a perfect treatment axis for free.
- mitochondrial, ribosomal, haemoglobin — the classic quality/ambient axes.
- `Xist`, `Malat1` — sex and nuclear-retention artifacts.

The M1/M2 question is answered by crosstab, not by eye: `M1_like_Mac` is exactly
`IFN_stimulated_Mac` in this annotation and `M2_like_Mac` is `Apoe_high` + `Arg1_high` + `Fn1_high`,
so "can we see the split" has a concrete test — **do Leiden clusters separate `IFN_stimulated_Mac`
from the three M2 states, and do the three M2 states separate from each other?**

In [ ]:
vn = adata.var_names
BLOCK = (set(vn[vn.str.startswith("mt-")]) | set(vn[vn.str.match(r"^Rp[ls]")])
         | set(vn[vn.str.match(r"^Hb[ab]-")]) | {"PSCA", "TAG72", "Xist", "Malat1"})
BLOCK &= set(vn)

sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat", batch_key="sample")
adata.var.loc[list(BLOCK), "highly_variable"] = False
hv = adata.var["highly_variable"].to_numpy()
print(f"HVG: {hv.sum():,} genes (blocked {len(BLOCK)} mito/ribo/Hb/spike-in/sex genes)")

work = adata[:, hv].copy()
sc.pp.scale(work, max_value=10)
sc.tl.pca(work, n_comps=N_PCS, svd_solver="arpack", random_state=SEED)
adata.obsm["X_pca_my"] = np.ascontiguousarray(work.obsm["X_pca"])
var_expl = work.uns["pca"]["variance_ratio"]
del work
print(f"PCA: {N_PCS} comps, {100*var_expl.sum():.1f}% of HVG variance")

import harmonypy

def harmonize(A, key="sample"):
    """Harmony on the myeloid PCs. harmonypy has returned Z_corr in BOTH orientations across
    versions (2.0.2 gives cells x PCs), and a silent transpose here would hand every downstream
    step a 50-cell object, so the orientation is resolved by shape and then asserted."""
    h = harmonypy.run_harmony(A.obsm["X_pca_my"], A.obs, [key],
                              max_iter_harmony=20, random_state=SEED)
    Z = np.asarray(h.Z_corr)
    if Z.shape[0] != A.n_obs:
        Z = Z.T
    assert Z.shape == (A.n_obs, A.obsm["X_pca_my"].shape[1]), \
        f"harmony returned {np.asarray(h.Z_corr).shape}, cannot align to {A.n_obs} cells"
    return np.ascontiguousarray(Z)

adata.obsm["X_hm"] = harmonize(adata)
print(f"harmony on 'sample' ({adata.obs['sample'].nunique()} libraries) -> X_hm "
      f"{adata.obsm['X_hm'].shape}")

sc.pp.neighbors(adata, n_neighbors=N_NEIGH, use_rep="X_hm", random_state=SEED)
sc.tl.umap(adata, random_state=SEED)
for r in (0.3, 0.6, 1.0):
    sc.tl.leiden(adata, resolution=r, key_added=f"leiden{r}", flavor="igraph",
                 n_iterations=2, directed=False, random_state=SEED)
    print(f"leiden {r}: {adata.obs[f'leiden{r}'].nunique()} clusters")

# ---- the M1/M2 question, answered by crosstab ------------------------------------------------
KEY = "leiden0.6"
ct = pd.crosstab(adata.obs[KEY], adata.obs.ct3).reindex(columns=LVL3_ORDER).fillna(0).astype(int)
frac = ct.div(ct.sum(1), axis=0)
summary = pd.DataFrame({
    "n": ct.sum(1),
    "dominant_state": frac.idxmax(1),
    "purity": frac.max(1).round(3),
})
print(f"\n=== {KEY}: which annotated state owns each cluster ===")
print(summary.sort_values("dominant_state").to_string())

M1, M2S = ["IFN_stimulated_Mac"], ["Apoe_high_Mac", "Arg1_high_Mac", "Fn1_high_Mac"]
own_m1 = set(summary.index[summary.dominant_state.isin(M1)])
own_m2 = set(summary.index[summary.dominant_state.isin(M2S)])
m2_states_owned = sorted(set(summary.loc[list(own_m2), "dominant_state"]))
# how much M1 leaks into M2-owned clusters and vice versa
leak_m1 = frac.loc[list(own_m2), M1].to_numpy().sum() / max(len(own_m2), 1)
leak_m2 = frac.loc[list(own_m1), M2S].to_numpy().sum() / max(len(own_m1), 1)
print(f"""
=== DOES CLUSTERING RECOVER THE M1/M2 SPLIT? ===
  clusters owned by M1 (IFN_stimulated_Mac) : {len(own_m1)}
  clusters owned by an M2 state             : {len(own_m2)}   states covered: {m2_states_owned}
  mean M1 fraction inside M2-owned clusters : {leak_m1:.3f}
  mean M2 fraction inside M1-owned clusters : {leak_m2:.3f}
  median cluster purity                     : {summary.purity.median():.3f}

  READ: >=1 M1-owned cluster AND all three M2 states owning at least one cluster, with both
        leak terms low, means the split is resolved and the M2 compartment is not one blob.
        If the three M2 states share a single cluster, M2 is a continuum at this resolution —
        which is itself the answer, and the reason a TRAJECTORY is the right tool for it.""")
adata.obs["ct3"] = pd.Categorical(adata.obs.ct3, categories=LVL3_ORDER, ordered=True)

### 3b. The map

Eight annotated states is past the three-slot all-pairs limit for a scatter, so identity is shown
as **small multiples** — one panel per state, that state in blue against the rest in grey —
rather than eight hues that no reader can separate under colour-vision deficiency. The two
continuous readouts (`SPP1_TAM_sig`, and Alice's independent binary call) get their own panels.

In [ ]:
U = adata.obsm["X_umap"]
present = [s for s in LVL3_ORDER if (adata.obs.ct3 == s).any()]
ncol = 5; nrow = int(np.ceil((len(present) + 3) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(2.35*ncol, 2.45*nrow))
axes = np.atleast_1d(axes).ravel()
PS = 0.9 if adata.n_obs > 40000 else 2.0

def _frame(ax, title, sub=None):
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_color(GRID)
    ax.set_title(title, fontsize=9, color=INK, loc="left", pad=3)
    if sub: ax.text(0.0, 1.005, sub, transform=ax.transAxes, fontsize=7.5, color=MUTED, va="bottom")

for ax, state in zip(axes, present):
    m = (adata.obs.ct3 == state).to_numpy()
    ax.scatter(U[~m, 0], U[~m, 1], s=PS, c=FADE, linewidths=0, rasterized=True)
    ax.scatter(U[m, 0],  U[m, 1],  s=PS, c=S1,   linewidths=0, rasterized=True)
    _frame(ax, state.replace("_", " "), f"n = {int(m.sum()):,}")

ax = axes[len(present)]
v = adata.obs.SPP1_TAM_sig.to_numpy()
sm = ax.scatter(U[:, 0], U[:, 1], s=PS, c=v, cmap=SEQ, linewidths=0, rasterized=True,
                vmin=np.quantile(v, .01), vmax=np.quantile(v, .99))
_frame(ax, "SPP1_TAM_sig (01c)", "graded, 50 genes, Spp1 excluded")
cb = fig.colorbar(sm, ax=ax, fraction=.045, pad=.02); cb.outline.set_visible(False)
cb.ax.tick_params(labelsize=7, colors=MUTED)

ax = axes[len(present) + 1]
m = (adata.obs.spp1_call == "Spp1_pos_mac").to_numpy()
ax.scatter(U[~m, 0], U[~m, 1], s=PS, c=FADE, linewidths=0, rasterized=True)
ax.scatter(U[m, 0],  U[m, 1],  s=PS, c=S2,   linewidths=0, rasterized=True)
_frame(ax, "Spp1_pos_mac (lvl4)", f"Alice's independent call, n = {int(m.sum()):,}")

ax = axes[len(present) + 2]
lab = adata.obs[KEY].astype(str)
ax.scatter(U[:, 0], U[:, 1], s=PS, c=FADE, linewidths=0, rasterized=True)
for cl in lab.unique():                                   # direct labels, no eight-hue scatter
    mm = (lab == cl).to_numpy()
    ax.text(np.median(U[mm, 0]), np.median(U[mm, 1]), cl, fontsize=7.5, color=INK,
            ha="center", va="center", fontweight="bold")
_frame(ax, f"{KEY} clusters", "directly labelled")

for ax in axes[len(present) + 3:]: ax.axis("off")
fig.suptitle("The myeloid manifold: annotated states, the SPP1-TAM score, and the clustering",
             fontsize=12, color=INK, x=0.005, ha="left", y=1.0)
fig.tight_layout()
fig.savefig(FIG/"12_umap_states.png", dpi=180, bbox_inches="tight")
plt.show(); plt.close(fig)

## 4. The root is a marker decision, taken before the tree is fitted

The root fixes the *direction* of every statement this notebook makes, so it cannot be chosen
after seeing the tree, and it cannot be chosen by anything gene-count-based (see the depth
argument in the header). It is chosen by **canonical markers**, scored here, and the choice is
committed to before §5 runs.

| programme | genes | what it marks |
|---|---|---|
| `classical_mono` | `Ly6c2 Plac8 Chil3 Sell Ccr2 F13a1 Hp Vcan Lyz2 Msrb1 S100a4` | Ly6C-high classical monocyte — **the expected root** |
| `patrolling_mono` | `Nr4a1 Ace Treml4 Cd300e Itgal Ear2 Fcgr4 Cx3cr1 Spn` | Ly6C-low patrolling monocyte — a **separate fate**, not an intermediate |
| `mature_mac` | `Adgre1 C1qa C1qb C1qc Mrc1 Csf1r Mafb Fcgr1 Mertk Cd68` | tissue-macrophage identity |
| `spp1_trem2_tam` | `Spp1 Trem2 Gpnmb Fabp5 Cd9 Lgals3 Ctsb Ctsd Ctsl Lpl Cd36 Atp6v0d2 Acp5` | the target terminal state |
| `ifn` | `Isg15 Ifit1 Ifit3 Irf7 Stat1 Gbp2 Gbp5 Cxcl9 Cxcl10 Rsad2 Oasl2` | the interferon axis — the axis `10` found the CAR contrast on |
| `cycle` | `Mki67 Top2a Ccnb1 Pcna Mcm5` | proliferation — a branch here is an artifact, not a fate |

`spp1_trem2_tam` is a **literature marker list**, and it shares genes with the 50-gene
`SPP1_TAM_sig`. That overlap is fine for *orienting* the tree — it is not fine for claiming which
genes drive the trajectory, so §7 removes the signature genes from every driver statement.

> ### The patrolling-monocyte trap (found by a trial fit, not reasoned about after the fact)
> A first fit rooted itself on the **`Patrolling_Mono` tip**, which put Ly6C-low monocytes
> *upstream* of classical monocytes — the reverse of the accepted direction. The cause is that a
> plain classical-monocyte score is not specific: `Lyz2`, `S100a4`, `Vcan` and `Ly6c2` itself are
> all appreciable in Ly6C-low monocytes, so the patrolling tip scores high on it.
>
> Two fixes, both applied below:
> 1. The root statistic is **`classical_mono − patrolling_mono`**, so a tip only wins by being
>    classical *and not* patrolling.
> 2. The root is restricted to nodes whose **dominant annotated state is a classical monocyte**
>    (`Phagocytic_Mono` or `IFN_stimulated_Mono`), and `tips_only=False` lets it sit at an
>    interior node — which is the biologically right answer if classical monocytes are the hub
>    that *both* the patrolling fate and the macrophage fate leave from. A tree rooted at an
>    interior node radiates in both directions, and `Patrolling_Mono` then gets its own branch,
>    which is what it should have.
>
> This matters beyond tidiness: the root sets the sign of every statement in §7 and §8.

In [ ]:
PROGRAMS = {
 "classical_mono":  ["Ly6c2","Plac8","Chil3","Sell","Ccr2","F13a1","Hp","Vcan","Lyz2","Msrb1","S100a4"],
 "patrolling_mono": ["Nr4a1","Ace","Treml4","Cd300e","Itgal","Ear2","Fcgr4","Cx3cr1","Spn"],
 "mature_mac":      ["Adgre1","C1qa","C1qb","C1qc","Mrc1","Csf1r","Mafb","Fcgr1","Mertk","Cd68"],
 "spp1_trem2_tam":  ["Spp1","Trem2","Gpnmb","Fabp5","Cd9","Lgals3","Ctsb","Ctsd","Ctsl","Lpl",
                     "Cd36","Atp6v0d2","Acp5"],
 "ifn":             ["Isg15","Ifit1","Ifit3","Irf7","Stat1","Gbp2","Gbp5","Cxcl9","Cxcl10",
                     "Rsad2","Oasl2"],
 "cycle":           ["Mki67","Top2a","Ccnb1","Pcna","Mcm5"],
}
for k, genes in PROGRAMS.items():
    present = [g for g in genes if g in adata.var_names]
    assert len(present) >= 4, f"{k}: only {len(present)} genes present"
    sc.tl.score_genes(adata, present, score_name=f"prog_{k}", random_state=SEED)
    if len(present) < len(genes):
        print(f"  {k}: {len(present)}/{len(genes)} genes present "
              f"(missing {sorted(set(genes)-set(present))})")

PCOLS = [f"prog_{k}" for k in PROGRAMS]
prof = adata.obs.groupby("ct3", observed=True)[PCOLS + ["SPP1_TAM_sig"]].mean().reindex(LVL3_ORDER)
print("=== programme scores by annotated state (mean) ===")
print(prof.round(3).to_string())

ROOT_PROGRAMME = "classical_mono"     # committed here, used unchanged in §5
top_root = prof[f"prog_{ROOT_PROGRAMME}"].idxmax()
top_term = prof["prog_spp1_trem2_tam"].idxmax()
print(f"""
ROOT COMMITMENT (made before the tree exists)
  root programme       : prog_{ROOT_PROGRAMME}
  highest-scoring state: {top_root}   ({prof.loc[top_root, f'prog_{ROOT_PROGRAMME}']:+.3f})
  expected terminal    : {top_term}   (prog_spp1_trem2_tam {prof.loc[top_term,'prog_spp1_trem2_tam']:+.3f})
  -> §5 puts the root at whichever TIP of the fitted tree has the highest mean
     prog_{ROOT_PROGRAMME}. The tree does not get to choose.""")

## 5. Fit the principal tree (scFates)

`scf.tl.tree(method="ppt")` fits a **SimplePPT** principal tree: `Nodes` points placed in a
low-dimensional space, connected as a minimum spanning tree, with cells softly assigned to nodes.
Two parameters control it — `sigma` (soft-assignment bandwidth; too small and the tree shatters
into spurious tips, too large and real branches collapse) and `lambda` (edge-length penalty).
`sigma` is chosen by the sweep below rather than by taste, and the chosen value's tip count is
reported so an over-branched fit is visible.

The tree is fitted in **Palantir's multiscale diffusion space** built on the harmonised PCs, which
is the representation scFates is designed for: diffusion components weight the manifold by
connectivity rather than by variance, so a rare intermediate state is not squeezed out by a
large one.

In [ ]:
import scFates as scf, palantir

# ---- multiscale diffusion space on the harmonised PCs ---------------------------------------
pcs = pd.DataFrame(adata.obsm["X_hm"], index=adata.obs_names)
dm  = palantir.utils.run_diffusion_maps(pcs, n_components=10)
ms  = palantir.utils.determine_multiscale_space(dm)
adata.obsm["X_ms"] = np.ascontiguousarray(ms.loc[adata.obs_names].to_numpy(), dtype=np.float64)
print(f"multiscale diffusion space: {adata.obsm['X_ms'].shape}")

# ---- sigma sweep: pick the smallest sigma whose tip count is stable -------------------------
if PPT_SIGMA is None:
    grid, rows = [0.05, 0.1, 0.15, 0.2, 0.3], []
    for s in grid:
        try:
            scf.tl.tree(adata, method="ppt", Nodes=N_NODES, use_rep="X_ms", device="cpu",
                        seed=SEED, ppt_lambda=PPT_LAMBDA, ppt_sigma=s, ppt_nsteps=100,
                        plot=False)
            rows.append(dict(sigma=s, n_tips=len(adata.uns["graph"]["tips"]),
                             n_forks=len(adata.uns["graph"]["forks"])))
        except Exception as e:
            rows.append(dict(sigma=s, n_tips=np.nan, n_forks=np.nan, err=str(e)[:60]))
    sweep = pd.DataFrame(rows)
    print("\n=== sigma sweep ===\n" + sweep.to_string(index=False))
    ok = sweep.dropna(subset=["n_tips"]).query("2 <= n_tips <= 8")
    PPT_SIGMA = float(ok.sigma.iloc[len(ok)//2]) if len(ok) else 0.1
    print(f"\n-> sigma = {PPT_SIGMA} (median of the sweep values giving 2-8 tips)")

scf.tl.tree(adata, method="ppt", Nodes=N_NODES, use_rep="X_ms", device="cpu",
            seed=SEED, ppt_lambda=PPT_LAMBDA, ppt_sigma=PPT_SIGMA, ppt_nsteps=200, plot=False)
# An edge between two branch points is a segment with no cells, and scf.tl.pseudotime cannot
# colour it — it raises "IndexError: index 0 is out of bounds for axis 0 with size 0". This is
# scFates' own remedy and is a no-op when there is nothing to merge.
scf.tl.merge_empty_segments(adata)
g = adata.uns["graph"]
print(f"\ntree: {g['F'].shape[1]} nodes | tips {list(np.asarray(g['tips']))} | "
      f"forks {list(np.asarray(g['forks']))}")

In [ ]:
# ---- which tip is the root? the §4 marker decision, applied ---------------------------------
# scFates writes the soft cell-to-node assignment to obsm['X_R'] (NOT uns['graph']['R']).
def soft_assign(a):
    Rm = a.obsm["X_R"]
    return np.asarray(Rm.todense()) if sp.issparse(Rm) else np.asarray(Rm)

R = soft_assign(adata)                                        # cells x nodes
W = R / R.sum(0, keepdims=True).clip(1e-12)                   # column-normalised -> node profile
tips = np.asarray(adata.uns["graph"]["tips"]).astype(int)
node_of_cell = R.argmax(1)

node_prof = pd.DataFrame(
    {c: W.T @ adata.obs[c].to_numpy(float) for c in PCOLS + ["SPP1_TAM_sig"]})
node_prof["n_cells"] = np.bincount(node_of_cell, minlength=W.shape[1])
tip_tab = node_prof.loc[tips].copy()
tip_tab["dominant_state"] = [adata.obs.ct3[node_of_cell == t].mode().iloc[0]
                             if (node_of_cell == t).any() else "—" for t in tips]
print("=== the tips, profiled ===")
print(tip_tab.round(3).to_string())

# The root statistic is classical MINUS patrolling — see the §4 trap — and the root is
# restricted to nodes a classical monocyte state actually owns. tips_only=False so the root may
# sit at an interior hub, which is correct if both fates leave from it.
node_prof["root_score"] = node_prof.prog_classical_mono - node_prof.prog_patrolling_mono
node_prof["dominant_state"] = [adata.obs.ct3[node_of_cell == i].mode().iloc[0]
                               if (node_of_cell == i).any() else "\u2014"
                               for i in range(node_prof.shape[0])]
node_prof["is_tip"] = node_prof.index.isin(tips)

cand = node_prof[(node_prof.n_cells >= MIN_CELLS_PER_NODE)
                 & (node_prof.dominant_state.isin(MONO_CLASSICAL))]
if len(cand):
    ROOT_NODE = int(cand.root_score.idxmax())
    why = f"node owned by a classical monocyte state ({len(cand)} candidates)"
else:
    ROOT_NODE = int(node_prof[node_prof.n_cells >= MIN_CELLS_PER_NODE].root_score.idxmax())
    why = ("NO node is owned by a classical monocyte state — fell back to the global max. "
           "Raise N_NODES or lower PPT_SIGMA and refit before trusting the direction.")
TERM_TIP = int(tip_tab["prog_spp1_trem2_tam"].idxmax())
print(f"""
root  -> node {ROOT_NODE}   {why}
         is_tip={bool(node_prof.loc[ROOT_NODE,'is_tip'])}, \
dominant state {node_prof.loc[ROOT_NODE,'dominant_state']}, \
n={int(node_prof.loc[ROOT_NODE,'n_cells'])}, root_score {node_prof.loc[ROOT_NODE,'root_score']:+.3f}
SPP1  -> tip {TERM_TIP}    highest prog_spp1_trem2_tam, \
dominant state {tip_tab.loc[TERM_TIP,'dominant_state']}""")
assert ROOT_NODE != TERM_TIP, "root and SPP1 tip are the same node — the tree has no axis. STOP."

scf.tl.root(adata, ROOT_NODE, tips_only=False)
scf.tl.pseudotime(adata, n_jobs=N_JOBS, n_map=N_MAP, seed=SEED)
adata.obs["t"] = adata.obs["t"].astype(float)
print(f"\npseudotime: {adata.obs.t.min():.3f} .. {adata.obs.t.max():.3f} | "
      f"{adata.obs.seg.nunique()} segments | milestones {sorted(adata.obs.milestones.unique())}")

order = (adata.obs.groupby("ct3", observed=True)["t"].agg(["mean", "median", "size"])
           .sort_values("mean"))
print("\n=== annotated states ordered by mean pseudotime ===")
print(order.round(3).to_string())
print(f"""
  READ: the ordering is the claim. Monocyte states at the low end and Apoe_high_Mac /
        Spp1_pos_mac at the high end is the trajectory the question posits. If Patrolling_Mono
        lands mid-trajectory rather than on its own tip, the tree has folded two lineages into
        one axis and the branch statements in §8 are not usable.""")

In [ ]:
# ---- the trajectory, three views -------------------------------------------------------------
# Pseudotime is magnitude -> one hue, light to dark. Segments are identity, but there are more
# than three, so they are direct-labelled on the tree rather than given their own hue scatter.
scf.tl.dendrogram(adata)

fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.6))
tt = adata.obs.t.to_numpy()

sm = axes[0].scatter(U[:, 0], U[:, 1], s=PS, c=tt, cmap=SEQ, linewidths=0, rasterized=True)
_frame(axes[0], "UMAP — pseudotime", "root fixed by prog_classical_mono")
cb = fig.colorbar(sm, ax=axes[0], fraction=.045, pad=.02); cb.outline.set_visible(False)
cb.set_label("pseudotime t", fontsize=8, color=MUTED); cb.ax.tick_params(labelsize=7, colors=MUTED)

D = adata.obsm["X_dendro"]
sm = axes[1].scatter(D[:, 0], D[:, 1], s=PS, c=tt, cmap=SEQ, linewidths=0, rasterized=True)
for msn in adata.obs.milestones.unique():                      # direct labels, not a hue per branch
    mm = (adata.obs.milestones == msn).to_numpy()
    axes[1].text(np.median(D[mm, 0]), np.median(D[mm, 1]), str(msn), fontsize=8,
                 color=INK, ha="center", va="center", fontweight="bold")
_frame(axes[1], "Tree space (dendrogram) — pseudotime", "milestones directly labelled")

# state composition along pseudotime: a stacked area is the part-to-whole form here
nb = 24
bins = pd.cut(adata.obs.t, nb, labels=False)
comp = (pd.crosstab(bins, adata.obs.ct3).reindex(columns=LVL3_ORDER).fillna(0))
comp = comp.div(comp.sum(1), axis=0)
xs = np.linspace(adata.obs.t.min(), adata.obs.t.max(), nb)
# ordered categories -> ordinal ramp; on a light surface it may not start lighter than
# step 250 (#86b6ef), which is 0.25 along this ramp.
shades = [SEQ(i) for i in np.linspace(0.25, 1.0, len(LVL3_ORDER))]
axes[2].stackplot(xs, comp[LVL3_ORDER].to_numpy().T, colors=shades, edgecolor=SURF, linewidth=1.2)
for i, s in enumerate(LVL3_ORDER):                              # direct label the widest band
    j = int(np.argmax(comp[s].to_numpy()))
    if comp[s].iloc[j] > 0.18:
        y = comp[LVL3_ORDER].iloc[j].cumsum()[s] - comp[s].iloc[j]/2
        axes[2].text(xs[j], y, s.replace("_", " "), fontsize=7.5, ha="center", va="center",
                     color="white" if i > len(LVL3_ORDER)/2 else INK)
axes[2].set_xlabel("pseudotime t", fontsize=9); axes[2].set_ylabel("fraction of cells", fontsize=9)
axes[2].set_ylim(0, 1); axes[2].set_xlim(xs[0], xs[-1])
axes[2].set_title("State composition along pseudotime", fontsize=9, color=INK, loc="left")
for s in ("top", "right"): axes[2].spines[s].set_visible(False)

fig.suptitle("The fitted trajectory", fontsize=12, color=INK, x=0.005, ha="left", y=1.02)
fig.tight_layout(); fig.savefig(FIG/"12_trajectory.png", dpi=180, bbox_inches="tight")
plt.show(); plt.close(fig)

## 6. Controls — the part that earns the trajectory

A principal tree will return a tree from pure noise. These four ask whether *this* one is
carrying biology. They run in increasing order of cost, and control 4 is the one that decides.

### Control 1 — is pseudotime a library-complexity gradient?

In [ ]:
# Depth and biology point the same way here (macrophages are deeper than monocytes), so the test
# is run WITHIN sample — where composition differences between animals cannot contribute — and
# calibrated against a marker axis measured the same way.
rows = []
for s, d in adata.obs.groupby("sample", observed=True):
    if len(d) < 100: continue
    rows.append(dict(sample=s, n=len(d),
        r_n_genes  = stats.spearmanr(d.t, d.n_genes_by_counts).statistic,
        r_total    = stats.spearmanr(d.t, d.total_counts).statistic,
        r_mature   = stats.spearmanr(d.t, d.prog_mature_mac).statistic,
        r_spp1     = stats.spearmanr(d.t, d.SPP1_TAM_sig).statistic))
dep = pd.DataFrame(rows)
print("=== within-sample Spearman of pseudotime vs depth, and vs the marker axes ===")
print(dep.describe().loc[["mean", "50%", "min", "max"]].round(3).to_string())

md_n, md_m = dep.r_n_genes.abs().median(), dep.r_mature.abs().median()
print(f"""
  |r| vs genes detected : {md_n:.3f}   (median over {len(dep)} samples)
  |r| vs mature_mac     : {md_m:.3f}
  ratio marker/depth    : {md_m/max(md_n,1e-9):.2f}x

  READ: a marker/depth ratio comfortably above 1 means the axis is biology that happens to
        correlate with depth. Below 1 means the reverse, and nothing downstream is usable.""")

# The decisive version: does the STATE ORDERING survive inside a depth stratum?
adata.obs["depth_q"] = pd.qcut(adata.obs.n_genes_by_counts, 4, labels=["Q1","Q2","Q3","Q4"])
per_q = (adata.obs.groupby(["depth_q", "ct3"], observed=True)["t"].mean()
           .unstack().reindex(columns=LVL3_ORDER))
print("\n=== mean pseudotime per state, WITHIN each quartile of genes detected ===")
print(per_q.round(3).to_string())
ref = per_q.mean(0).rank()
cons = {q: stats.spearmanr(per_q.loc[q].rank(), ref, nan_policy="omit").statistic for q in per_q.index}
print("\n  Spearman of each quartile's state ordering vs the pooled ordering:")
for q, r in cons.items(): print(f"    {q}: rho = {r:+.3f}")
print(f"""
  READ: rho near +1 in all four quartiles means the ordering of the annotated states is the
        same at every sequencing depth — the strongest available evidence that the axis is not
        the depth artifact that 09 caught. A quartile that inverts is a red flag.""")
dep.to_csv(OUT/"ctrl1_depth_within_sample.csv", index=False)

### Control 2 — is pseudotime a batch axis, and does it survive a different embedding?

In [ ]:
def eta2(values, groups):
    """One-way variance explained (eta squared) — the share of pseudotime a factor accounts for."""
    v = np.asarray(values, float); g = pd.Series(np.asarray(groups)).astype(str)
    grand = v.mean()
    ssb = sum(len(v[g.values == k]) * (v[g.values == k].mean() - grand) ** 2 for k in g.unique())
    return float(ssb / ((v - grand) ** 2).sum())

print("=== share of pseudotime variance explained, one factor at a time ===")
facts = {"cell state (ct3)": adata.obs.ct3, "sample (= library = mouse-tumour)": adata.obs["sample"],
         "mouse": adata.obs.mouse, "condition": adata.obs.cond, "CAR background": adata.obs.cart,
         "tumour site": adata.obs.tu}
e2 = {k: eta2(adata.obs.t, v) for k, v in facts.items()}
for k, v in sorted(e2.items(), key=lambda kv: -kv[1]): print(f"  {k:36s} {100*v:5.1f}%")
print(f"""
  READ: cell state must dominate. 'sample' above 'cell state' means harmony did not remove the
        batch axis and the trajectory is partly a library ordering. Note sample == mouse-tumour
        == library here (1:1 over all 29), so this term cannot be decomposed further.""")

# ---- the same ordering from a different embedding and a different algorithm -------------------
# DPT on the object's INHERITED whole-object harmony PCs. Nothing is shared with §5 except the
# root cell, so agreement is evidence the ordering is a property of the data, not of the pipeline.
# §2 loads only X/obs/var, so the inherited embedding is pulled from the h5ad here — which also
# keeps this control self-contained and re-runnable on its own.
with h5py.File(SC_REF, "r") as f:
    _has_inh = ("obsm" in f) and ("X_pca_harmony" in f["obsm"])
    if _has_inh:
        _names = pd.Index(ad.io.read_elem(f["obs"][f["obs"].attrs["_index"]]).astype(str))
        _H = np.asarray(ad.io.read_elem(f["obsm"]["X_pca_harmony"]))

if not _has_inh:
    print("SKIPPED — the reference carries no obsm['X_pca_harmony'], so there is no inherited\n"
          "embedding to cross-check against. Controls 1, 3 and 4 are unaffected.")
    r_all = r_ord = float("nan")
else:
    _pos = _names.get_indexer(adata.obs_names.astype(str))
    assert (_pos >= 0).all(), "some cells are not in the reference index — cannot align"
    # a bare shell carrying only the inherited embedding — DPT needs no expression matrix
    aux = ad.AnnData(X=np.zeros((adata.n_obs, 1), dtype=np.float32),
                     obs=pd.DataFrame(index=adata.obs_names),
                     obsm={"X_pca_harmony": np.ascontiguousarray(_H[_pos])})
    del _H
    sc.pp.neighbors(aux, n_neighbors=N_NEIGH, use_rep="X_pca_harmony", random_state=SEED)
    sc.tl.diffmap(aux, n_comps=15)
    aux.uns["iroot"] = int(np.argmin(adata.obs.t.to_numpy()))     # the same root cell as §5
    sc.tl.dpt(aux)
    adata.obs["t_dpt_inherited"] = aux.obs["dpt_pseudotime"].to_numpy()
    del aux

    r_all = stats.spearmanr(adata.obs.t, adata.obs.t_dpt_inherited).statistic
    ord_scf = adata.obs.groupby("ct3", observed=True)["t"].mean().reindex(LVL3_ORDER)
    ord_dpt = adata.obs.groupby("ct3", observed=True)["t_dpt_inherited"].mean().reindex(LVL3_ORDER)
    r_ord = stats.spearmanr(ord_scf.rank(), ord_dpt.rank(), nan_policy="omit").statistic
    print(f"""
=== scFates (re-embedded, PPT tree) vs DPT (inherited X_pca_harmony, diffusion) ===
  per-cell Spearman            : {r_all:+.3f}
  state-ordering Spearman (n={ord_scf.notna().sum()}) : {r_ord:+.3f}
  state ordering, side by side:
{pd.DataFrame({'scFates_t': ord_scf, 'DPT_inherited': ord_dpt}).round(3).to_string()}

  READ: the per-cell number will be lower than the state-level one and that is expected — DPT has
        no tree and smears the branches together. The STATE ORDERING is the comparable quantity.""")

### Control 3 — is a branch actually proliferation, ambient RNA, or dying cells?

In [ ]:
qc = ["prog_cycle", "pct_counts_mt", "n_genes_by_counts", "doublet_score", "pct_counts_ribo"]
qc = [c for c in qc if c in adata.obs.columns]
adata.obs["t_bin"] = pd.qcut(adata.obs.t, 10, labels=[f"D{i+1}" for i in range(10)])
print("=== QC along pseudotime deciles ===")
print(adata.obs.groupby("t_bin", observed=True)[qc].mean().round(3).to_string())
print("\n=== QC per milestone (the branch tips) ===")
print(adata.obs.groupby("milestones", observed=True)[qc + ["t"]]
        .agg(["mean"]).round(3).droplevel(1, axis=1).to_string())
print(f"""
  READ: a milestone that is the proliferation score and nothing else is a cell-cycle branch, not
        a fate, and must be excluded from §8. A milestone with high pct_counts_mt and low
        n_genes is a dying-cell branch. Either one invalidates the branch statements, not the
        main axis.""")

### Control 4 — does the whole thing reproduce in both CAR-T backgrounds?

This is the project's standing rule (`08b`, `09`, `10`, `rtAdvantage/01`): **two independent
backgrounds, and the evidence is that they agree**, not a p-value from one fit. The entire
pipeline — HVG, PCA, harmony, diffusion space, tree, root, pseudotime — is rebuilt from scratch
inside the PSCA arms and inside the TAG72 arms, which never see each other's cells.

Three things are compared, cheap to expensive:

1. **Topology** — tip count, and which annotated state owns each tip.
2. **State ordering** — Spearman between the two backgrounds' orderings of the 8 states.
3. **Gene level** — each gene's Spearman with pseudotime, computed separately in each background,
   then the correlation *between* the two vectors, against a **label permutation null**. `09`'s
   lesson applies directly: a parametric p on 10,000 correlated genes is meaningless (it gave
   p = 1.7e-152 where the permutation gave 0.49), so the null is permuted.

In [ ]:
# scFates keeps trajectory state in .uns/.obsm/.obs, and a subset COPIES all of it. Refitting on
# top of that stale state makes scf.tl.pseudotime fail deep inside: it builds seg_colors by
# matching uns['milestones_colors'] (from the PARENT fit) against the NEW tree's milestones,
# matches nothing, and indexes an empty array —
#     IndexError: index 0 is out of bounds for axis 0 with size 0
# So every refit starts from a clean slate.
TRAJ_UNS  = ["graph", "ppt", "epg", "pseudotime_list", "milestones_colors", "seg_colors",
             "dendro_segments", "seg", "milestones"]
TRAJ_OBSM = ["X_R", "X_ms", "X_dendro"]
TRAJ_OBS  = ["t", "seg", "milestones", "edge"]

def reset_traj(a):
    """Strip every trace of a previous scFates fit, in place."""
    for k in TRAJ_UNS:  a.uns.pop(k, None)
    for k in TRAJ_OBSM: a.obsm.pop(k, None)
    for k in TRAJ_OBS:  a.obs.drop(columns=k, inplace=True, errors="ignore")
    for k in [k for k in a.uns if "<>" in str(k)]: a.uns.pop(k, None)   # fork results
    return a

def fit_tree_safe(a, n_nodes, sigma=None, tag="", tip_range=(2, 12)):
    # 12 rather than 8: the biology admits ~5-6 legitimate ends (patrolling monocyte,
    # Apoe/SPP1, Arg1, Fn1, Mmp12, a cycling branch) and forcing a tighter range pushes
    # sigma to extreme values that over-smooth the tree rather than fixing it.
    """Fit a principal tree that scf.tl.pseudotime can actually consume.

    Three failure modes, all seen on real subsets of this data:

    1. EMPTY SEGMENTS. An edge joining two branch points carries no cells. scFates assigns
       milestones only to segments that have cells, so that endpoint never reaches
       uns['graph']['milestones'] and building seg_colors indexes an empty array:
           IndexError: index 0 is out of bounds for axis 0 with size 0
       tl.merge_empty_segments is scFates' own remedy; it must run BEFORE tl.root, because
       root builds pp_seg from the graph.
    2. A SHATTERED TREE. sigma is not transferable between manifolds - the value §5 picked on
       all 79k cells gave 36 tips on an 11k-cell background subset, with scFates itself warning
       "not converged / more than 30 tips detected". So sigma is re-swept HERE, per object,
       unless the caller pins it.
    3. OVER-NODING, which is what creates the adjacent forks in (1) to begin with; the node
       count is stepped down if a fit still cannot be turned into a pseudotime.
    """
    sigmas = [sigma] if sigma is not None else [0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 0.8]
    for nn in [n_nodes, int(n_nodes * 0.6), int(n_nodes * 0.35)]:
        scored = []
        for sg in sigmas:
            reset_traj(a)
            try:
                scf.tl.tree(a, method="ppt", Nodes=nn, use_rep="X_ms", device="cpu", seed=SEED,
                            ppt_lambda=PPT_LAMBDA, ppt_sigma=sg, ppt_nsteps=200, plot=False)
                scf.tl.merge_empty_segments(a)
                scored.append((sg, len(np.asarray(a.uns["graph"]["tips"]))))
            except Exception as e:
                scored.append((sg, np.nan))
        ok = [(sg, k) for sg, k in scored if np.isfinite(k) and tip_range[0] <= k <= tip_range[1]]
        # prefer a tip count inside the range; otherwise the fit closest to its middle
        target = sum(tip_range) / 2
        order = (sorted(ok, key=lambda x: x[0]) if ok else
                 sorted([t for t in scored if np.isfinite(t[1])], key=lambda x: abs(x[1] - target)))
        if not order:
            continue
        print(f"    [{tag}] Nodes={nn} sigma sweep -> "
              + ", ".join(f"{sg}:{int(k) if np.isfinite(k) else 'x'} tips" for sg, k in scored))
        for sg, k in order:
            reset_traj(a)
            scf.tl.tree(a, method="ppt", Nodes=nn, use_rep="X_ms", device="cpu", seed=SEED,
                        ppt_lambda=PPT_LAMBDA, ppt_sigma=sg, ppt_nsteps=200, plot=False)
            scf.tl.merge_empty_segments(a)
            try:
                scf.tl.root(a, _pick_root(a)[0], tips_only=False)
                scf.tl.pseudotime(a, n_jobs=N_JOBS, n_map=1, seed=SEED)
            except IndexError:
                print(f"    [{tag}] sigma={sg} still left an empty segment - next candidate")
                continue
            if not ok:
                print(f"    [{tag}] WARNING: no sigma gave {tip_range[0]}-{tip_range[1]} tips; "
                      f"used sigma={sg} with {int(k)} tips. Topology comparisons below are weak.")
            return a, nn, sg, int(k)
    raise RuntimeError(f"[{tag}] no (Nodes, sigma) gave a tree pseudotime could consume")


def _pick_root(a):
    """The §4 marker rule, factored out so the tree fitter can reuse it."""
    Rm = soft_assign(a); Wm = Rm / Rm.sum(0, keepdims=True).clip(1e-12); noc = Rm.argmax(1)
    tab = pd.DataFrame({c: Wm.T @ a.obs[c].to_numpy(float) for c in PCOLS + ["SPP1_TAM_sig"]})
    tab["n_cells"] = np.bincount(noc, minlength=Wm.shape[1])
    tab["dominant_state"] = [a.obs.ct3[noc == i].mode().iloc[0] if (noc == i).any() else "-"
                             for i in range(tab.shape[0])]
    tab["root_score"] = tab.prog_classical_mono - tab.prog_patrolling_mono
    tab["is_tip"] = tab.index.isin(np.asarray(a.uns["graph"]["tips"]).astype(int))
    ok = tab[(tab.n_cells >= MIN_CELLS_PER_NODE) & (tab.dominant_state.isin(MONO_CLASSICAL))]
    pool = ok if len(ok) else tab[tab.n_cells >= MIN_CELLS_PER_NODE]
    return int(pool.root_score.idxmax()), tab, len(ok) > 0


def fit_trajectory(sub, tag, n_nodes=None, sigma=None):
    """The whole §3+§5 pipeline, self-contained, on a subset. Returns (obs_frame, tips, adata)."""
    sub = reset_traj(sub.copy())
    sc.pp.highly_variable_genes(sub, n_top_genes=2000, flavor="seurat", batch_key="sample")
    sub.var.loc[list(BLOCK & set(sub.var_names)), "highly_variable"] = False
    w = sub[:, sub.var.highly_variable.to_numpy()].copy()
    sc.pp.scale(w, max_value=10); sc.tl.pca(w, n_comps=N_PCS, svd_solver="arpack", random_state=SEED)
    sub.obsm["X_pca_my"] = np.ascontiguousarray(w.obsm["X_pca"]); del w
    sub.obsm["X_hm"] = harmonize(sub)
    p = pd.DataFrame(sub.obsm["X_hm"], index=sub.obs_names)
    m = palantir.utils.determine_multiscale_space(palantir.utils.run_diffusion_maps(p, n_components=10))
    sub.obsm["X_ms"] = np.ascontiguousarray(m.loc[sub.obs_names].to_numpy(), dtype=np.float64)
    # node count scaled to the subset: 500 nodes over 26k cells is far denser than over 79k,
    # and over-noding is exactly what produces the adjacent forks that break pseudotime.
    # Node count scaled to the subset (500 nodes over 26k cells is far denser than over 79k),
    # and sigma re-swept per background rather than inherited from §5 — see fit_tree_safe.
    nn = n_nodes or max(100, int(round(N_NODES * sub.n_obs / adata.n_obs)))
    sub, nn_used, sg_used, n_tips = fit_tree_safe(sub, nn, sigma, tag)
    rn, allp, classical_owned = _pick_root(sub)
    tp = np.asarray(sub.uns["graph"]["tips"]).astype(int)
    prof_s = allp.loc[tp, PCOLS + ["SPP1_TAM_sig", "n_cells", "dominant_state"]]
    print(f"  [{tag}] {sub.n_obs:,} cells | Nodes={nn_used} sigma={sg_used} | {len(tp)} tips | "
          f"root node {rn} ({allp.loc[rn,'dominant_state']}, classical-owned: {classical_owned})")
    return sub.obs[["ct3", "t", "milestones", "sample"]].copy(), prof_s, sub

print("refitting the entire pipeline per CAR background (this is the expensive control)\n")
BG_FITS = {}
for bg in ("PSCA", "TAG72"):
    sub = adata[(adata.obs.cart == bg).to_numpy()]
    BG_FITS[bg] = fit_trajectory(sub, bg)

print("\n=== 1. topology ===")
for bg, (_, tp, _) in BG_FITS.items():
    print(f"\n  [{bg}] tips:")
    print(tp[[f"prog_{ROOT_PROGRAMME}", "prog_spp1_trem2_tam", "SPP1_TAM_sig",
              "dominant_state"]].round(3).to_string().replace("\n", "\n    "))

print("\n=== 2. state ordering ===")
ords = {bg: ob.groupby("ct3", observed=True)["t"].mean().reindex(LVL3_ORDER)
        for bg, (ob, _, _) in BG_FITS.items()}
ords["full"] = adata.obs.groupby("ct3", observed=True)["t"].mean().reindex(LVL3_ORDER)
O = pd.DataFrame(ords)
print(O.round(3).to_string())
r_bg = stats.spearmanr(O.PSCA.rank(), O.TAG72.rank(), nan_policy="omit").statistic
print(f"\n  PSCA vs TAG72 state-ordering Spearman: {r_bg:+.3f}")

print("\n=== 3. gene level, with a permutation null ===")
det = np.asarray((adata.X > 0).mean(0)).ravel()
TEST_GENES = adata.var_names[(det > DETECT_FLOOR) & ~adata.var_names.isin(BLOCK)]
print(f"  genes tested: {len(TEST_GENES):,} (detected in >{100*DETECT_FLOOR:.0f}% of cells)")

def gene_t_corr(sub_adata, t, block=1000):
    """Spearman(expression, pseudotime) per gene — rank once, then a vectorised Pearson.

    Done in blocks of genes: densifying 79k x 8.6k at once costs ~2.7 GB, and ranking it
    promotes to float64 for another ~5.4 GB. Blocking keeps the peak near 1 GB at no real
    cost in speed, since the work is the ranking either way."""
    Rt = stats.rankdata(np.asarray(t, float)); Rt = Rt - Rt.mean()
    denom_t = np.sqrt((Rt ** 2).sum())
    out = np.empty(len(TEST_GENES), dtype=np.float64)
    cols = sub_adata.var_names.get_indexer(TEST_GENES)
    for i in range(0, len(cols), block):
        sl = cols[i:i + block]
        Xg = sub_adata.X[:, sl]
        Xg = Xg.toarray() if sp.issparse(Xg) else np.asarray(Xg)
        Rx = stats.rankdata(Xg, axis=0); Rx -= Rx.mean(0)
        out[i:i + block] = ((Rx * Rt[:, None]).sum(0)
                            / (np.sqrt((Rx ** 2).sum(0)) * denom_t + 1e-12))
        del Xg, Rx
    return pd.Series(out, index=TEST_GENES)

GC = {}
for bg, (ob, _, sub) in BG_FITS.items():
    GC[bg] = gene_t_corr(sub, ob.t.to_numpy())
GC = pd.DataFrame(GC).dropna()
r_obs = stats.pearsonr(GC.PSCA, GC.TAG72).statistic
null = np.array([stats.pearsonr(GC.PSCA, rng.permutation(GC.TAG72.to_numpy())).statistic
                 for _ in range(N_PERM)])
p_perm = (1 + (np.abs(null) >= abs(r_obs)).sum()) / (1 + N_PERM)
GC.to_csv(OUT/"ctrl4_gene_pseudotime_corr_by_background.csv")
print(f"""
  cross-background r of gene-vs-pseudotime correlation: {r_obs:+.3f}
  label permutation null: mean {null.mean():+.3f}, |r| 95th pct {np.quantile(np.abs(null),0.95):.3f}
  permutation p = {p_perm:.4f}   ({N_PERM:,} permutations)

  BENCHMARKS from the same project, through comparable machinery:
    treated vs NoTx, pseudobulk logFC, cross-background   r = +0.89  (09 §7)
    RT contrast, the known null                           r = -0.29, perm p = 0.49
  READ: an r in the +0.7..+0.9 band puts this trajectory alongside the project's positive
        control. An r near 0 with a non-significant permutation p puts it alongside the RT null,
        and the trajectory is then a per-background artifact that must not be pooled.""")

## 7. What changes along the trajectory

`scf.tl.test_association` fits a GAM of each gene against pseudotime and keeps the ones with a
significant, large-enough effect; `scf.tl.fit` produces the smoothed trend per gene;
`scf.tl.cluster` groups those trends into **modules** — genes that turn on together, which is the
unit worth interpreting.

> **The circularity fence.** The 50 genes of `SPP1_TAM_sig` are what oriented the tree in §4-5.
> They are dropped from the module tables and from every enrichment below, so no statement here
> can be "the genes we used to define the endpoint are high at the endpoint". The pathway sets are
> stripped of them too, exactly as `09` strips them.

**Runtime, measured rather than guessed.** A trial run on 8,604 cells × 4,149 genes at
`n_jobs=8` took **18 minutes**, split `test_association` 4:05 → **`fit` 13:28** → `cluster` 0:24.
`scf.tl.fit` is the long pole, not the significance test, and cost is roughly linear in
cells × genes. Scaling that to this object (~11k testable genes) on 6 performance cores:

| | cells fitted | rough estimate |
|---|---|---|
| `GAM_CELL_CAP = 15000` (default) | 15,000 | **~1.5 h** |
| `GAM_CELL_CAP = 25000` | 25,000 | ~2.5 h |
| `GAM_CELL_CAP = None` | 79,000 | ~8 h |

The cap subsamples **evenly across milestones**, so no branch is thinned relative to another, and
the quantities being estimated are population-level curves — 15,000 cells is ~1,900 per annotated
state, far more than a spline needs. Set it to `None` only if a module assignment looks unstable.

> **`scf.tl.fit` subsets the object in place** — afterwards `gfit` holds only the significant
> features and the unfiltered data moves to `gfit.raw`. So the association table is written out
> *before* that call; otherwise the non-significant genes' `fdr`/`A` are lost and the denominator
> silently becomes the numerator.

In [ ]:
# scFates fits its GAMs through R's mgcv via rpy2, so this needs the traj_r_py311 kernel.
# The work is done on a GENE-SUBSET COPY: the graph, obs and obsm come along with the slice,
# and it keeps the fitted-trend layer off the 79k x 23k parent object.
adata.var["tested"] = adata.var_names.isin(TEST_GENES)
gfit = adata[:, adata.var.tested.to_numpy()].copy()

# One mgcv GAM per gene per cell is the most expensive thing in this project. Cost scales with
# genes x cells, and the trends being estimated are population-level curves, so a stratified
# subsample costs almost no precision and turns hours into minutes. Cells are drawn evenly
# across milestones so no branch is thinned relative to the others. Set GAM_CELL_CAP = None to
# fit on all cells; the module assignments should not change materially either way.
if GAM_CELL_CAP and gfit.n_obs > GAM_CELL_CAP:
    per = max(1, GAM_CELL_CAP // gfit.obs.milestones.nunique())
    take = (gfit.obs.groupby("milestones", observed=True)
                .apply(lambda d: d.sample(min(len(d), per), random_state=SEED).index)
                .explode().dropna().to_numpy())
    gfit = gfit[take].copy()
    print(f"GAM fits on a milestone-stratified subsample: {gfit.n_obs:,} of {adata.n_obs:,} cells "
          f"(<= {per:,} per milestone)")
    print(f"  state coverage retained: "
          f"{gfit.obs.ct3.value_counts().reindex(LVL3_ORDER).fillna(0).astype(int).to_dict()}")
print(f"testing {gfit.n_vars:,} genes against pseudotime "
      f"(GAM per gene via mgcv, {N_JOBS} jobs — this is the long step)")
scf.tl.test_association(gfit, n_jobs=N_JOBS, A_cut=0.3, fdr_cut=1e-4, spline_df=5)
sig_genes = gfit.var_names[gfit.var["signi"].astype(bool)].tolist()
print(f"significantly associated with pseudotime: {len(sig_genes):,} / {gfit.n_vars:,}")
assert sig_genes, "no gene is associated with pseudotime — loosen A_cut/fdr_cut, or the axis is noise"

# The association table has to be taken NOW: scf.tl.fit SUBSETS the object to the significant
# features ("adata subsetted to keep only fitted features!"), so afterwards the non-significant
# genes' p_val/fdr/A are gone. Saving here keeps the full table, which is the honest denominator.
ASSOC = gfit.var[[c for c in ("A", "p_val", "fdr", "st", "signi") if c in gfit.var.columns]].copy()
ASSOC.to_csv(OUT/"gene_pseudotime_association.csv")
for c in ASSOC.columns:
    adata.var[c] = ASSOC[c].reindex(adata.var_names)

scf.tl.fit(gfit, n_jobs=N_JOBS)          # the long pole: ~3x test_association
scf.tl.cluster(gfit, n_neighbors=20, metric="correlation")
MODCOL = next(c for c in ("clusters", "fit_clusters") if c in gfit.var.columns)
adata.var[MODCOL] = gfit.var[MODCOL].reindex(adata.var_names)
print(f"after fit(): gfit is subsetted to {gfit.n_vars:,} fitted features "
      f"(unfiltered data kept in gfit.raw)")
mods = gfit.var.loc[sig_genes, MODCOL].astype(str)
print(f"\nmodules ({MODCOL}): {mods.nunique()}")
print(mods.value_counts().sort_index().to_string())

SIGSET = set(SIG_PRESENT)
mod_tab = []
for m in sorted(mods.unique(), key=lambda x: int(x) if str(x).isdigit() else 999):
    genes = [g for g in mods.index[mods == m]]
    clean = [g for g in genes if g not in SIGSET]
    if not clean: continue
    trend = np.asarray(gfit[:, clean].layers["fitted"]).mean(1)
    peak_t = float(gfit.obs.t.to_numpy()[np.argsort(trend)[-max(50, len(trend)//50):]].mean())
    r_t = stats.spearmanr(trend, gfit.obs.t).statistic
    mod_tab.append(dict(module=m, n_genes=len(genes), n_after_fence=len(clean),
                        dropped_signature=len(genes) - len(clean),
                        peak_pseudotime=peak_t, rho_with_t=r_t,
                        top=", ".join(clean[:8])))
MOD = pd.DataFrame(mod_tab).sort_values("peak_pseudotime")
MOD.to_csv(OUT/"module_summary.csv", index=False)
print("\n=== modules, ordered by where they peak ===")
print(MOD.drop(columns="top").round(3).to_string(index=False))
print("\n  leading genes per module (signature genes already removed):")
for _, r in MOD.iterrows(): print(f"    {r.module:>3}  {r.top}")
pd.DataFrame({"gene": mods.index, "module": mods.values}).to_csv(
    OUT/"gene_modules.csv", index=False)

In [ ]:
# ---- GSEA (NES) on the gene-vs-pseudotime ranking: the 01c §4d engine, verbatim ---------------
from statsmodels.stats.multitest import multipletests
N_PERM_GSEA, CHUNK = 10000, 1000

def _es_at_hits(pos, wcum, Wtot, N, n):
    i = np.arange(1, n + 1)
    phit_after  = wcum / Wtot
    phit_before = np.concatenate([np.zeros(pos.shape[:-1] + (1,)), wcum[..., :-1]], axis=-1) / Wtot
    dev = np.concatenate([phit_after  - (pos + 1 - i) / (N - n),
                          phit_before - (pos - i + 1) / (N - n)], axis=-1)
    hi, lo = np.maximum(dev.max(axis=-1), 0.0), np.minimum(dev.min(axis=-1), 0.0)
    return np.where(hi >= -lo, hi, lo)

def gsea_prerank(ranking, gene_sets, n_perm=N_PERM_GSEA, weight=1.0, seed=SEED,
                 min_size=5, max_size=500, chunk=CHUNK):
    """Classic weighted-KS GSEA with a gene-set-permutation null (as fgsea / GSEA-preranked)."""
    r = ranking.dropna().sort_values(ascending=False)
    w, N = np.abs(r.to_numpy(float)) ** weight, len(r)
    idx = pd.Series(np.arange(N), index=r.index)
    _rng = np.random.default_rng(seed)
    rows, null_nes = [], []
    for name, members in gene_sets.items():
        m = idx.index.intersection(pd.Index(sorted(set(members))))
        n = len(m)
        if not (min_size <= n <= max_size): continue
        pos = np.sort(idx.loc[m].to_numpy())
        es = float(_es_at_hits(pos, np.cumsum(w[pos]), w[pos].sum(), N, n))
        null, done = np.empty(n_perm), 0
        while done < n_perm:
            b = min(chunk, n_perm - done)
            perm = np.sort(_rng.random((b, N)).argpartition(n, axis=1)[:, :n], axis=1)
            null[done:done+b] = _es_at_hits(perm, np.cumsum(w[perm], axis=1),
                                            w[perm].sum(axis=1)[:, None], N, n)
            done += b
        pn, nn_ = null[null >= 0], null[null < 0]
        if es >= 0: nes_v, p = es / pn.mean(),  (1 + (pn >= es).sum()) / (1 + pn.size)
        else:       nes_v, p = -es / nn_.mean(), (1 + (nn_ <= es).sum()) / (1 + nn_.size)
        rows.append({"raw": name, "n_genes": n, "ES": es, "NES": float(nes_v), "pval": float(p)})
        null_nes.append(np.where(null >= 0, null / pn.mean(), -null / nn_.mean()))
    out = pd.DataFrame(rows)
    out["FDR_BH"] = multipletests(out["pval"], method="fdr_bh")[1]
    pool, obs_ = np.concatenate(null_nes), out["NES"].to_numpy()
    q = np.empty(len(obs_))
    for i, v in enumerate(obs_):
        if v >= 0: q[i] = (pool[pool >= 0] >= v).mean() / max((obs_[obs_ >= 0] >= v).mean(), 1e-12)
        else:      q[i] = (pool[pool < 0] <= v).mean() / max((obs_[obs_ < 0] <= v).mean(), 1e-12)
    out["FDR"] = np.minimum(q, 1.0)
    return out.sort_values("NES", ascending=False).reset_index(drop=True)

def read_gmt(p):
    out = {}
    for line in open(p):
        x = line.rstrip("\n").split("\t")
        if len(x) >= 3: out[x[0]] = x[2:]
    return out
gene_sets = {}
for f in ["mh.all.v2026.1.Mm.symbols.gmt", "m2.cp.reactome.v2026.1.Mm.symbols.gmt",
          "m5.go.bp.v2026.1.Mm.symbols.gmt", "m2.cp.v2026.1.Mm.symbols.gmt"]:
    gene_sets.update(read_gmt(GMT_DIR/f))

FOCUS = ["REACTOME_BINDING_AND_UPTAKE_OF_LIGANDS_BY_SCAVENGER_RECEPTORS", "GOBP_PHAGOCYTOSIS",
         "GOBP_LYSOSOMAL_TRANSPORT", "GOBP_LIPID_DROPLET_ORGANIZATION", "GOBP_CHOLESTEROL_EFFLUX",
         "GOBP_CHOLESTEROL_STORAGE", "GOBP_LIPID_OXIDATION", "GOBP_FATTY_ACID_BETA_OXIDATION",
         "GOBP_MACROPHAGE_DERIVED_FOAM_CELL_DIFFERENTIATION", "GOBP_LONG_CHAIN_FATTY_ACID_IMPORT_INTO_CELL",
         "GOBP_CELLULAR_RESPONSE_TO_OXIDISED_LOW_DENSITY_LIPOPROTEIN_PARTICLE_STIMULUS",
         "GOBP_ADENOSINE_METABOLIC_PROCESS", "GOBP_MONOCYTE_DIFFERENTIATION",
         "GOBP_MACROPHAGE_DIFFERENTIATION", "GOBP_MYELOID_CELL_DIFFERENTIATION"]
panel = {k: v for k, v in gene_sets.items() if k.startswith("HALLMARK_")}
for gs in FOCUS:
    if gs in gene_sets: panel[gs] = gene_sets[gs]
    else: print("  missing from GMT:", gs)
# the circularity fence: signature genes out of every set, so no set can win on them
panel = {k: [g for g in v if g not in SIGSET] for k, v in panel.items()}

t_corr = gene_t_corr(adata, adata.obs.t.to_numpy()).drop(labels=[g for g in SIG_PRESENT
                                                                if g in TEST_GENES], errors="ignore")
nes = gsea_prerank(t_corr, panel)
nes.to_csv(OUT/"gsea_along_pseudotime.csv", index=False)
print(f"\n=== GSEA on Spearman(gene, pseudotime); + = rises toward the SPP1 end ===")
print(f"  {len(nes)} sets tested, {int((nes.FDR < 0.05).sum())} at FDR<0.05\n")
print("  TOP 15 UP\n" + nes.head(15)[["raw","n_genes","NES","FDR"]].round(3).to_string(index=False))
print("\n  TOP 10 DOWN\n" + nes.tail(10)[["raw","n_genes","NES","FDR"]].round(3).to_string(index=False))

In [ ]:
# ---- module trends: a diverging ramp, because z-scored trend is polarity (above/below the mean)
ORDER = MOD.module.tolist()
bins = pd.qcut(gfit.obs.t, 40, labels=False)   # gfit may be the §7 subsample
rows, labels = [], []
for m in ORDER:
    genes = [g for g in mods.index[mods == m] if g not in SIGSET]
    if not genes: continue
    tr = np.asarray(gfit[:, genes].layers["fitted"]).mean(1)
    prof_m = pd.Series(tr).groupby(bins.to_numpy()).mean().to_numpy()
    rows.append((prof_m - prof_m.mean()) / (prof_m.std() + 1e-9))
    labels.append(f"module {m}  (n={len(genes)})")
Z = np.vstack(rows)

fig, ax = plt.subplots(figsize=(9.5, 0.42*len(Z) + 2.2))
im = ax.imshow(Z, aspect="auto", cmap=DIV, vmin=-2.2, vmax=2.2,
               extent=[gfit.obs.t.min(), gfit.obs.t.max(), len(Z)-0.5, -0.5])
ax.set_yticks(range(len(Z))); ax.set_yticklabels(labels, fontsize=8.5, color=INK)
ax.set_xlabel("pseudotime t  →  toward the SPP1/TREM2 end", fontsize=9)
ax.set_xlim(gfit.obs.t.min(), gfit.obs.t.max())
ax.tick_params(length=0)
for s in ax.spines.values(): s.set_color(GRID)
cb = fig.colorbar(im, ax=ax, fraction=.022, pad=.015)
cb.set_label("module trend (z across pseudotime)", fontsize=8, color=MUTED)
cb.outline.set_visible(False); cb.ax.tick_params(labelsize=7, colors=MUTED)
ax.set_title("Gene modules switch on in sequence along the trajectory",
             fontsize=12, color=INK, loc="left", pad=10)
fig.text(0.005, -0.01, "Modules ordered by where they peak. Signature genes removed from every "
         "module before averaging (the §7 circularity fence).", fontsize=7.5, color=MUTED)
fig.tight_layout(); fig.savefig(FIG/"12_module_trends.png", dpi=180, bbox_inches="tight")
plt.show(); plt.close(fig)

## 8. The fork — what separates the SPP1 branch from its sibling

`scf.tl.test_fork` tests, for each gene, whether its trend differs between the two branches
downstream of a fork; `branch_specific` keeps the genes that are genuinely one-sided;
`activation` asks **where in pseudotime** each one switches on, which is the part that says
whether the SPP1/TREM2 program is an early commitment or a late elaboration.

Run only if §6 control 3 cleared the milestones of being cell-cycle or dying-cell branches, and
only on the fork that actually separates the SPP1 tip from a sibling.

In [ ]:
milestones = adata.obs.milestones.astype(str)
ms_prof = (adata.obs.assign(ms=milestones)
             .groupby("ms", observed=True)[["t", "SPP1_TAM_sig"] + PCOLS].mean())
ms_prof["n"] = milestones.value_counts()
ms_prof["dominant_state"] = (adata.obs.assign(ms=milestones)
                             .groupby("ms", observed=True)["ct3"]
                             .agg(lambda s: s.mode().iloc[0] if len(s) else "—"))
ms_prof["cycle_branch"] = ms_prof.prog_cycle > CYCLE_FLAG          # control 3, made operational
print("=== milestones ===")
print(ms_prof.sort_values("t").round(3).to_string())
if ms_prof.cycle_branch.any():
    print(f"\n  EXCLUDED as proliferation branches (prog_cycle > {CYCLE_FLAG}): "
          f"{list(ms_prof.index[ms_prof.cycle_branch])}")

# ---- the fork is read off the TREE, not guessed from pseudotime ------------------------------
# uns['graph']['pp_seg'] is the segment table (from -> to over milestone nodes), already oriented
# away from the root by scf.tl.root. So the sibling of the SPP1 branch is a graph question.
seg = adata.uns["graph"]["pp_seg"]
edges = [(str(r["from"]), str(r["to"])) for _, r in seg.iterrows()]
children, parent = {}, {}
for a, b in edges:
    children.setdefault(a, []).append(b); parent[b] = a
ROOT_MS = str(adata.uns["graph"]["root"])
leaves = [m for m in ms_prof.index if m not in children]

def leaves_below(m):
    out, stack = [], [m]
    while stack:
        x = stack.pop()
        (stack.extend(children[x]) if x in children else out.append(x))
    return out

ok_leaves = [m for m in leaves if not ms_prof.loc[m, "cycle_branch"]]
assert ok_leaves, "every leaf is a proliferation branch — nothing to test"
SPP1_MS = ms_prof.loc[ok_leaves, "SPP1_TAM_sig"].idxmax()
print(f"\nroot milestone : {ROOT_MS}  ({ms_prof.loc[ROOT_MS,'dominant_state']})")
print(f"leaves         : {leaves}  (usable: {ok_leaves})")
print(f"SPP1 leaf      : {SPP1_MS}  (SPP1_TAM_sig {ms_prof.loc[SPP1_MS,'SPP1_TAM_sig']:+.3f}, "
      f"{ms_prof.loc[SPP1_MS,'dominant_state']})")

path = [SPP1_MS]
while path[-1] in parent: path.append(parent[path[-1]])
path = path[::-1]
branch_nodes = [x for x in path[:-1] if len(children.get(x, [])) > 1]

# The deepest fork is the one to prefer, but its sibling may be the proliferation branch that
# control 3 disqualified. So walk back up the path and take the deepest fork that still has a
# usable sibling, and say how far back that was.
FORK, cand, on_path = None, [], None
for f in reversed(branch_nodes):
    op = path[path.index(f) + 1]
    c_ = [l for ch in children[f] if ch != op
          for l in leaves_below(ch) if not ms_prof.loc[l, "cycle_branch"]]
    if c_:
        FORK, cand, on_path = f, c_, op
        break
    print(f"  fork {f}: sibling branch(es) {[ch for ch in children[f] if ch != op]} are all "
          f"proliferation branches — stepping back toward the root")

if not branch_nodes:
    print("\nNO FORK on the path from the root to the SPP1 leaf — that branch is a plain curve. "
          "§8 does not apply, which is a result rather than a failure.")
else:
    if FORK is None:
        print("\nEvery fork on the root->SPP1 path has only proliferation branches as siblings. "
              "There is nothing legitimate to contrast the SPP1 branch against. §8 skipped.")
    else:
        depth_back = len(branch_nodes) - 1 - branch_nodes.index(FORK)
        if depth_back:
            print(f"  using a fork {depth_back} level(s) back from the deepest one")
        PARTNER = ms_prof.loc[cand, "n"].idxmax()          # the biggest usable sibling leaf
        PAIR = [str(SPP1_MS), str(PARTNER)]
        print(f"fork           : {FORK}  ({ms_prof.loc[FORK,'dominant_state']}), "
              f"children {children[FORK]}")
        print(f"fork partner   : {PARTNER}  ({ms_prof.loc[PARTNER,'dominant_state']}, "
              f"n={int(ms_prof.loc[PARTNER,'n'])})  chosen from {cand}")
        print(f"\ntesting fork {FORK} -> {PAIR}")
        scf.tl.test_fork(gfit, root_milestone=str(FORK), milestones=PAIR,
                         n_jobs=N_JOBS, rescale=True)
        scf.tl.branch_specific(gfit, root_milestone=str(FORK), milestones=PAIR, effect=0.3)
        scf.tl.activation(gfit, root_milestone=str(FORK), milestones=PAIR,
                          deriv_cut=0.15, n_jobs=N_JOBS)
        fork_key = next(k for k in gfit.uns if "<>" in str(k))
        fk = gfit.uns[fork_key]["fork"].copy()
        fk = fk[~fk.index.isin(SIGSET)]                     # the circularity fence again

        # The fork table has NO "effect" column: the two branch amplitudes are columns named
        # after the milestones themselves, and a gene is assigned to the branch where it sits
        # relatively higher. So the signed magnitude is their difference.
        fk["delta"] = fk[PAIR[0]].astype(float) - fk[PAIR[1]].astype(float)   # + = SPP1 branch
        fk.to_csv(OUT/"fork_genes.csv")
        print(f"\nbranch-specific genes (signature genes removed): {len(fk):,}  "
              f"[branch_specific effect cut = 0.3]")
        for b in PAIR:
            sb = fk[fk.branch.astype(str) == b].reindex(
                     fk.delta.abs().sort_values(ascending=False).index).dropna(subset=["branch"])
            print(f"\n  toward {b} ({ms_prof.loc[b,'dominant_state']}, {len(sb)} genes), "
                  f"top 20 by |delta|:")
            print("    " + ", ".join(sb.index[:20]))
            if len(sb) and "activation" in sb.columns:
                print(f"    median activation pseudotime {sb.activation.median():.3f} "
                      f"(fork at t = {ms_prof.loc[FORK,'t']:.3f}, leaf at "
                      f"{ms_prof.loc[b,'t']:.3f})")
            if len(sb) and "module" in sb.columns:
                em = sb["module"].astype(str).value_counts().to_dict()
                print(f"    early / late split: {em}")
        print(f"""
  READ (this is what §8 is for): 'late' means the programme switches on AFTER the branch point —
  an elaboration of a fate already chosen. 'early' means it is on at or before the fork — part of
  the commitment itself. If the SPP1/TREM2 genes are overwhelmingly 'late', the SPP1-TAM state is
  something macrophages acquire once they are already on that branch, not what puts them on it.""")
        if len(fk):
            nes_fork = gsea_prerank(fk["delta"], panel)
            nes_fork.to_csv(OUT/"gsea_fork.csv", index=False)
            print(f"\n=== GSEA on the fork contrast (+ = toward {PAIR[0]}, the SPP1 branch) ===")
            print(nes_fork.head(12)[["raw","n_genes","NES","FDR"]].round(3).to_string(index=False))
            print("   ...")
            print(nes_fork.tail(8)[["raw","n_genes","NES","FDR"]].round(3).to_string(index=False))

## 9. Save, and the summary block

In [ ]:
keep_obs = ["ct2","ct3","cond","sample","tu","mouse","cart","RT","spp1_call","isg15_call",
            "SPP1_TAM_sig","t","seg","milestones","edge","t_dpt_inherited","depth_q","t_bin",  # t_dpt_inherited is optional
            KEY,"n_genes_by_counts","total_counts","pct_counts_mt"] + PCOLS
keep_obs = [c for c in keep_obs if c in adata.obs.columns]
adata.obs[keep_obs].to_parquet(OUT/"myeloid_trajectory_cells.parquet")

TRAJ_H5 = OUT/"myeloid_trajectory.h5ad"
for k in list(adata.layers): del adata.layers[k]        # none expected; X is all 13 needs
adata.write_h5ad(TRAJ_H5, compression="gzip")
gfit.write_h5ad(OUT/"myeloid_trajectory_fitted_genes.h5ad", compression="gzip")
print(f"wrote {TRAJ_H5}  ({TRAJ_H5.stat().st_size/1e9:.2f} GB) — notebook 13 reads this")
print(f"wrote {OUT/'myeloid_trajectory_fitted_genes.h5ad'} (tested genes + fitted trends + fork)")

print("\n" + "="*96)
print("SUMMARY — report these back")
print("="*96)
ordr = adata.obs.groupby("ct3", observed=True)["t"].mean().reindex(LVL3_ORDER).dropna()
print(f"""
THE TRAJECTORY
  cells                 : {adata.n_obs:,} myeloid ({', '.join(LADDER)})
  tree                  : {len(np.asarray(adata.uns['graph']['tips']))} tips, sigma {PPT_SIGMA}, {N_NODES} nodes
  root                  : node {ROOT_TIP}, chosen by prog_{ROOT_PROGRAMME} BEFORE fitting
  state order by t      : {' < '.join(ordr.index.tolist())}
  SPP1 end confirmed by : SPP1_TAM_sig {adata.obs.groupby('milestones', observed=True)['SPP1_TAM_sig'].mean().max():+.3f} at the top milestone,
                          and Alice's independent lvl4 Spp1_pos_mac rate along t (below)

CONTROLS
  1 depth   : |rho| t-vs-genes {md_n:.3f} vs t-vs-mature_mac {md_m:.3f} (ratio {md_m/max(md_n,1e-9):.2f}x);
              state ordering inside depth quartiles rho = {', '.join(f'{v:+.2f}' for v in cons.values())}
  2 batch   : variance explained — state {100*e2['cell state (ct3)']:.1f}%, sample {100*e2['sample (= library = mouse-tumour)']:.1f}%;
              independent embedding+method (DPT on inherited PCs) state-order rho = {r_ord:+.3f}
  3 QC      : see the per-milestone table — a cycle-only or mito-high milestone invalidates §8 only
  4 reprod. : PSCA vs TAG72, state-order rho = {r_bg:+.3f}; gene-vs-t correlation r = {r_obs:+.3f},
              permutation p = {p_perm:.4f}  (project benchmarks: +0.89 positive, -0.29/p=0.49 null)

WHAT MOVES ALONG IT
  genes associated with t : {len(sig_genes):,} / {len(TEST_GENES):,} tested
  modules                 : {len(MOD)}, peaking from t={MOD.peak_pseudotime.min():.2f} to {MOD.peak_pseudotime.max():.2f}
  top pathway UP toward SPP1 : {nes.iloc[0]['raw']}  NES {nes.iloc[0]['NES']:+.2f}, FDR {nes.iloc[0]['FDR']:.3g}
  top pathway DOWN           : {nes.iloc[-1]['raw']}  NES {nes.iloc[-1]['NES']:+.2f}, FDR {nes.iloc[-1]['FDR']:.3g}

WORDING THIS LICENSES
  * "myeloid transcriptomes in these tumours lie on a reproducible ordering whose ends are
     classical monocyte and SPP1/TREM2 TAM" — IF control 4's r is in the positive-control band.
  * NOT "monocytes differentiate into SPP1-TAMs in this tumour". One dissociated endpoint
     snapshot, no velocity, no lineage tracing. The ordering is transcriptional, the direction
     is a marker decision, and the word for it is ORDERING, not differentiation.
  * Condition effects on this axis are notebook 13's, not this one's.

FILES -> {OUT}
  myeloid_trajectory.h5ad          (notebook 13 input)
  myeloid_trajectory_cells.parquet
  ctrl1_depth_within_sample.csv | ctrl4_gene_pseudotime_corr_by_background.csv
  module_summary.csv | gene_modules.csv | gsea_along_pseudotime.csv | gsea_fork.csv | fork_genes.csv
  figures/12_umap_states.png | 12_trajectory.png | 12_module_trends.png""")